# Lintgate Complete Suite

**Auto-generated** — Complete suite: ALL analyses combined

- **Target repo**: `https://github.com/rohanvinaik/Wayfinder.git`
- **Branch**: `main`
- **Source dirs**: `['src']`
- **Mutation profiling**: `True`
- **Architecture**: External project
- **Tool**: LintGate (`https://github.com/rohanvinaik/LintGate.git`)
- **Target**: `https://github.com/rohanvinaik/Wayfinder.git`

**How to use:** Runtime > Run all. Download the zip at the end.
Pass the `action_plan.json` to any LLM coding agent.

The action plan is:
- **Prioritized**: P0 blocking → P1 critical → P2 important → P3 improve
- **Dependency-ordered**: lint fixes before spec work, tests before mutation profiling
- **Actionable**: each item has specific tool commands and expected outcomes


## Step 1: Install LintGate + Clone Target


In [ ]:
import importlib, os, shutil, subprocess, sys

TARGET_REPO_URL = "https://github.com/rohanvinaik/Wayfinder.git"
TARGET_BRANCH = "main"
LINTGATE_REPO_URL = "https://github.com/rohanvinaik/LintGate.git"
LINTGATE_DIR = "/content/lintgate"
PROJECT_DIR = "/content/project"
SRC_DIRS = ['src']

# Uncomment if repos are private:
# GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"

def _clone(url, branch, dest):
    if os.path.exists(dest):
        shutil.rmtree(dest)
    clone_url = url
    try:
        GITHUB_TOKEN
        clone_url = url.replace("https://", f"https://{GITHUB_TOKEN}@")
    except NameError:
        pass
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', branch, clone_url, dest],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        result = subprocess.run(
            ['git', 'clone', '--depth', '1', clone_url, dest],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(f'Clone {url} failed: {result.stderr}')
    print(f'Cloned {url} → {dest}')

# Step 1: Clone and install LintGate (the analysis tool)
print('Installing LintGate...')
_clone(LINTGATE_REPO_URL, 'prescriptive-spec-system', LINTGATE_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'hatchling', 'pyyaml', 'packaging'], check=True)

# Try pip install first (best), fall back to sys.path (always works)
pip_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', LINTGATE_DIR],
    capture_output=True, text=True
)
if pip_result.returncode != 0:
    print(f'pip install -e failed (expected on some Colab runtimes): {pip_result.stderr[:200]}')
    print('Falling back to sys.path injection...')

# Always ensure LINTGATE_DIR is on sys.path (belt and suspenders)
if LINTGATE_DIR not in sys.path:
    sys.path.insert(0, LINTGATE_DIR)

# Step 2: Clone the target project
print('Cloning target project...')
_clone(TARGET_REPO_URL, TARGET_BRANCH, PROJECT_DIR)

# Purge stale module cache and verify import
for m in list(sys.modules):
    if m.startswith('lintgate') or m.startswith('mcp_tools'):
        del sys.modules[m]
importlib.invalidate_caches()

# Verify LintGate is importable
try:
    import lintgate
    print(f'LintGate package at: {lintgate.__file__}')
except ImportError as e:
    print(f'ERROR: Cannot import lintgate: {e}')
    print(f'sys.path: {sys.path[:5]}')
    print(f'LINTGATE_DIR contents: {os.listdir(LINTGATE_DIR)[:10]}')
    raise

print(f'Target project at {PROJECT_DIR}')
print(f'Source dirs to analyze: ['src']')


## Step 2: Run Full Analysis


In [ ]:
import json, os, time

INCLUDE_MUTATION = True
MUTATION_BUDGET_MS = 500

from lintgate.offline_analysis import run_complete_analysis

print('Running: Complete suite: ALL analyses combined')
print()

result = run_complete_analysis(
    PROJECT_DIR,
    src_dirs=['src'],
    include_mutation=INCLUDE_MUTATION,
)

print(f'Analysis complete in {result["elapsed_s"]}s')
print(f'Source files: {result["project"]["total_source_files"]}')
print(f'Test files: {result["project"]["total_test_files"]}')
print(f'Total LoC: {result["project"]["total_loc"]}')
print(f'Action items: {len(result["action_plan"])}')
if 'lint' in result:
    print(f'Lint findings: {result["lint"]["total_findings"]}')
if 'specification' in result:
    print(f'Functions analyzed: {result["specification"]["total_functions"]}')
if 'coherence' in result:
    print(f'Coherence: {result["coherence"]["state"]}')
if 'convergence_roadmap' in result and isinstance(result['convergence_roadmap'], list):
    print(f'Convergence targets: {len(result["convergence_roadmap"])}')


## Step 3: Review Action Plan


In [ ]:
print('=' * 80)
print('ACTION PLAN — prioritized fixes for LLM implementation')
print('=' * 80)

for action in result['action_plan']:
    rank = action['rank']
    priority = action['priority']
    cat = action['category']
    f = action.get('file', '')
    deps = action.get('depends_on', [])
    effort = action.get('estimated_effort', '')
    print(f'\n[{rank}] {priority} | {cat} | {effort}')
    print(f'    File: {f}')
    if action.get('function'):
        print(f'    Function: {action["function"]}')
    print(f'    Action: {action["action"]}')
    if deps:
        print(f'    Depends on: {deps}')

print(f'\n{"=" * 80}')
p_counts = {}
for a in result['action_plan']:
    p = a['priority']
    p_counts[p] = p_counts.get(p, 0) + 1
for p, n in sorted(p_counts.items()):
    print(f'  {p}: {n}')


## Step 4: Save & Download


In [ ]:
import json, os

# Save the full analysis artifact
output_path = os.path.join(PROJECT_DIR, '.lintgate', 'full_analysis.json')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(result, f, indent=2)
size_kb = os.path.getsize(output_path) / 1024
print(f'Analysis saved: {output_path} ({size_kb:.0f} KB)')

# Also save a compact action-plan-only file
plan_path = os.path.join(PROJECT_DIR, '.lintgate', 'action_plan.json')
with open(plan_path, 'w') as f:
    json.dump({
        'project': result['project']['name'],
        'timestamp': result['timestamp'],
        'summary': {
            'source_files': result['project']['total_source_files'],
            'lint_findings': result['lint']['total_findings'],
            'under_specified': result['specification']['under_specified_count'],
            'action_count': len(result['action_plan']),
        },
        'action_plan': result['action_plan'],
    }, f, indent=2)
print(f'Action plan saved: {plan_path}')

# Zip for download
zip_path = '/content/lintgate_analysis.zip'
!cd {PROJECT_DIR} && zip -r {zip_path} .lintgate/full_analysis.json .lintgate/action_plan.json -x '*.DS_Store'

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'\nDownload ready: {size_mb:.1f} MB')
    try:
        from google.colab import files
        files.download(zip_path)
        print('Download started!')
    except ImportError:
        print(f'Not in Colab. Results at: {zip_path}')
else:
    print('No zip created.')

print(f'\n--- HOW TO USE ---')
print(f'Pass the action_plan.json to your LLM coding agent with:')
print(f'  "Implement the fixes in this action plan, starting from rank 1."')
print(f'  "Each action has priority, dependencies, and specific instructions."')
print(f'  "Apply results to: /Users/rohanvinaik/Projects/Wayfinder"')
